In [7]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType
)
# import findspark
# findspark.init()

import os
from pyspark.sql import SparkSession

# Initialize your local Spark Session
spark = SparkSession.builder \
    .appName("StockDataIngestion") \
    .master("local[*]") \
    .getOrCreate()

print("⚡ Spark Session Created Successfully!")

⚡ Spark Session Created Successfully!


In [8]:
stock_schema = StructType([
    StructField("Symbol", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Last Sale", StringType(), True),
    StructField("Net Change", DoubleType(), True),
    StructField("% Change", StringType(), True),
    StructField("Market Cap", DoubleType(), True),
    StructField("Country", StringType(), True),
    StructField("IPO Year", LongType(), True),
    StructField("Volume", LongType(), True),
    StructField("Sector", StringType(), True),
    StructField("Industry", StringType(), True)
])

In [9]:
# file_path = os.path.abspath('../../22-07-26/nasdaq_screener.csv')
# print(file_path)
df = spark.read.csv(
    '../../22-07-26/nasdaq_screener.csv',
    header=True,
    schema=stock_schema
)

In [10]:
df.printSchema()

root
 |-- Symbol: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Last Sale: string (nullable = true)
 |-- Net Change: double (nullable = true)
 |-- % Change: string (nullable = true)
 |-- Market Cap: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- IPO Year: long (nullable = true)
 |-- Volume: long (nullable = true)
 |-- Sector: string (nullable = true)
 |-- Industry: string (nullable = true)



In [11]:
df.show()

+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+
|Symbol|                Name|Last Sale|Net Change|% Change|     Market Cap|       Country|IPO Year|  Volume|              Sector|            Industry|
+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+
|     A|Agilent Technolog...|  $132.86|       2.6|  1.996%| 3.752390808E10| United States|    1999| 2383955|         Industrials|Biotechnology: La...|
|    AA|Alcoa Corporation...|   $44.35|      0.87|  2.001%|1.1703515956E10| United States|    2016| 6846054|         Industrials|            Aluminum|
|  AACB|Artius II Acquisi...|   $10.51|      0.01|  0.095%|            0.0| United States|    2025|   28508|                NULL|                NULL|
| AACBR|Artius II Acquisi...|    $0.18|       0.0|   0.00%|            0.0| United States|    

In [12]:
from pyspark.sql.functions import regexp_replace, col

df = df.withColumns({"Last Sale": regexp_replace(col("Last Sale"), r"\$", "").cast("double"),
                    "% Change": regexp_replace(col("% Change"), r"%", "").cast("double")})

In [13]:
df.show()

+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+
|Symbol|                Name|Last Sale|Net Change|% Change|     Market Cap|       Country|IPO Year|  Volume|              Sector|            Industry|
+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+
|     A|Agilent Technolog...|   132.86|       2.6|   1.996| 3.752390808E10| United States|    1999| 2383955|         Industrials|Biotechnology: La...|
|    AA|Alcoa Corporation...|    44.35|      0.87|   2.001|1.1703515956E10| United States|    2016| 6846054|         Industrials|            Aluminum|
|  AACB|Artius II Acquisi...|    10.51|      0.01|   0.095|            0.0| United States|    2025|   28508|                NULL|                NULL|
| AACBR|Artius II Acquisi...|     0.18|       0.0|     0.0|            0.0| United States|    

In [18]:
df.filter((col("Market Cap") != 0) &\
          (col("% Change") > 0) &\
          (col("Country").isNotNull())).show()

+------+--------------------+---------+----------+--------+----------------+--------------+--------+--------+--------------------+--------------------+
|Symbol|                Name|Last Sale|Net Change|% Change|      Market Cap|       Country|IPO Year|  Volume|              Sector|            Industry|
+------+--------------------+---------+----------+--------+----------------+--------------+--------+--------+--------------------+--------------------+
|     A|Agilent Technolog...|   132.86|       2.6|   1.996|  3.752390808E10| United States|    1999| 2383955|         Industrials|Biotechnology: La...|
|    AA|Alcoa Corporation...|    44.35|      0.87|   2.001| 1.1703515956E10| United States|    2016| 6846054|         Industrials|            Aluminum|
|   AAL|American Airlines...|    15.28|      0.14|   0.925| 1.0105964893E10| United States|    NULL|90810300|Consumer Discreti...|Air Freight/Deliv...|
|  AAMI|Acadian Asset Man...|    83.95|       1.8|   2.191|   2.991053543E9|United Kingd

In [19]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+------+----+---------+----------+--------+----------+-------+--------+------+------+--------+
|Symbol|Name|Last Sale|Net Change|% Change|Market Cap|Country|IPO Year|Volume|Sector|Industry|
+------+----+---------+----------+--------+----------+-------+--------+------+------+--------+
|     0|   0|        0|         0|       1|       398|    306|    2991|     0|   746|     747|
+------+----+---------+----------+--------+----------+-------+--------+------+------+--------+



In [23]:
df_clean = df.dropna(subset=["Country", "% Change"])

In [27]:
df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
]).show()

+------+----+---------+----------+--------+----------+-------+--------+------+------+--------+
|Symbol|Name|Last Sale|Net Change|% Change|Market Cap|Country|IPO Year|Volume|Sector|Industry|
+------+----+---------+----------+--------+----------+-------+--------+------+------+--------+
|     0|   0|        0|         0|       0|         0|      0|       0|     0|   690|     691|
+------+----+---------+----------+--------+----------+-------+--------+------+------+--------+



In [25]:
df_clean.show()

+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+
|Symbol|                Name|Last Sale|Net Change|% Change|     Market Cap|       Country|IPO Year|  Volume|              Sector|            Industry|
+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+
|     A|Agilent Technolog...|   132.86|       2.6|   1.996| 3.752390808E10| United States|    1999| 2383955|         Industrials|Biotechnology: La...|
|    AA|Alcoa Corporation...|    44.35|      0.87|   2.001|1.1703515956E10| United States|    2016| 6846054|         Industrials|            Aluminum|
|  AACB|Artius II Acquisi...|    10.51|      0.01|   0.095|            0.0| United States|    2025|   28508|                NULL|                NULL|
| AACBR|Artius II Acquisi...|     0.18|       0.0|     0.0|            0.0| United States|    

In [28]:
df_clean = df_clean.fillna({"Market Cap": 0, "IPO Year": 2026, "Sector": "Unknown", "Industry": "Unknown"})
df_clean.show()

+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+
|Symbol|                Name|Last Sale|Net Change|% Change|     Market Cap|       Country|IPO Year|  Volume|              Sector|            Industry|
+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+
|     A|Agilent Technolog...|   132.86|       2.6|   1.996| 3.752390808E10| United States|    1999| 2383955|         Industrials|Biotechnology: La...|
|    AA|Alcoa Corporation...|    44.35|      0.87|   2.001|1.1703515956E10| United States|    2016| 6846054|         Industrials|            Aluminum|
|  AACB|Artius II Acquisi...|    10.51|      0.01|   0.095|            0.0| United States|    2025|   28508|             Unknown|             Unknown|
| AACBR|Artius II Acquisi...|     0.18|       0.0|     0.0|            0.0| United States|    

In [29]:
df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
]).show()

+------+----+---------+----------+--------+----------+-------+--------+------+------+--------+
|Symbol|Name|Last Sale|Net Change|% Change|Market Cap|Country|IPO Year|Volume|Sector|Industry|
+------+----+---------+----------+--------+----------+-------+--------+------+------+--------+
|     0|   0|        0|         0|       0|         0|      0|       0|     0|     0|       0|
+------+----+---------+----------+--------+----------+-------+--------+------+------+--------+



In [32]:
from pyspark.sql.functions import round
df_clean.withColumn("Open", round(col("Last Sale") - col("Net Change"), 2)).show()

+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+------+
|Symbol|                Name|Last Sale|Net Change|% Change|     Market Cap|       Country|IPO Year|  Volume|              Sector|            Industry|  Open|
+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+------+
|     A|Agilent Technolog...|   132.86|       2.6|   1.996| 3.752390808E10| United States|    1999| 2383955|         Industrials|Biotechnology: La...|130.26|
|    AA|Alcoa Corporation...|    44.35|      0.87|   2.001|1.1703515956E10| United States|    2016| 6846054|         Industrials|            Aluminum| 43.48|
|  AACB|Artius II Acquisi...|    10.51|      0.01|   0.095|            0.0| United States|    2025|   28508|             Unknown|             Unknown|  10.5|
| AACBR|Artius II Acquisi...|     0.18|       0.0|  

In [33]:
df_clean = df_clean.withColumn("Open", round(col("Last Sale") - col("Net Change"), 2))
df_clean.show()

+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+------+
|Symbol|                Name|Last Sale|Net Change|% Change|     Market Cap|       Country|IPO Year|  Volume|              Sector|            Industry|  Open|
+------+--------------------+---------+----------+--------+---------------+--------------+--------+--------+--------------------+--------------------+------+
|     A|Agilent Technolog...|   132.86|       2.6|   1.996| 3.752390808E10| United States|    1999| 2383955|         Industrials|Biotechnology: La...|130.26|
|    AA|Alcoa Corporation...|    44.35|      0.87|   2.001|1.1703515956E10| United States|    2016| 6846054|         Industrials|            Aluminum| 43.48|
|  AACB|Artius II Acquisi...|    10.51|      0.01|   0.095|            0.0| United States|    2025|   28508|             Unknown|             Unknown|  10.5|
| AACBR|Artius II Acquisi...|     0.18|       0.0|  

In [ ]:
from pyspark.sql.functions import avg, sum, max, min, count

df_agg = df_clean.groupBy("IPO Year").agg(
    avg("Last Sale").alias("Avg Price"),
    max("Last Sale").alias("Max Price"),
    min("Last Sale").alias("Min Price"),
    max("Market Cap").alias("Max Market Cap"),
    count("*").alias("Stock Count"),
    min("Volume").alias("Min Volume")
).orderBy("IPO Year")

df_agg.show(100)

+--------+------------------+---------+---------+-----------------+-----------+----------+
|IPO Year|         Avg Price|Max Price|Min Price|   Max Market Cap|Stock Count|Min Volume|
+--------+------------------+---------+---------+-----------------+-----------+----------+
|    1925|             42.15|    42.15|    42.15|  1.0329909838E11|          1|   4182801|
|    1929|            124.22|   124.22|   124.22|   6.007192246E10|          1|    421778|
|    1930|              9.75|     9.75|     9.75|        9419787.0|          1|       964|
|    1946|              6.76|     6.76|     6.76|      4.4280941E7|          1|      5388|
|    1951|             68.91|    84.72|     53.1|   1.285153812E10|          2|     58128|
|    1960| 96.02666666666666|   137.66|    58.73|  1.6529454819E10|          3|    162645|
|    1965|              1.03|     1.03|     1.03|      1.4827427E7|          1|     48816|
|    1968|              0.21|     0.21|     0.21|      2.3072844E7|          1|    256536|

In [41]:
from pyspark.sql.functions import floor
df_agg = df_agg.withColumn("Avg Price", round(col("Avg Price"), 2)) \
    .withColumn("Century", (floor(col("IPO Year") / 100, 0) + 1).cast("int"))

df_agg.show(100)

+--------+---------+---------+---------+-----------------+-----------+----------+-------+
|IPO Year|Avg Price|Max Price|Min Price|   Max Market Cap|Stock Count|Min Volume|Century|
+--------+---------+---------+---------+-----------------+-----------+----------+-------+
|    1925|    42.15|    42.15|    42.15|  1.0329909838E11|          1|   4182801|     20|
|    1929|   124.22|   124.22|   124.22|   6.007192246E10|          1|    421778|     20|
|    1930|     9.75|     9.75|     9.75|        9419787.0|          1|       964|     20|
|    1946|     6.76|     6.76|     6.76|      4.4280941E7|          1|      5388|     20|
|    1951|    68.91|    84.72|     53.1|   1.285153812E10|          2|     58128|     20|
|    1960|    96.03|   137.66|    58.73|  1.6529454819E10|          3|    162645|     20|
|    1965|     1.03|     1.03|     1.03|      1.4827427E7|          1|     48816|     20|
|    1968|     0.21|     0.21|     0.21|      2.3072844E7|          1|    256536|     20|
|    1969|

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank
from pyspark.sql.functions import floor
df_agg = df_agg.withColumn("Avg Price", round(col("Avg Price"), 2)) \
    .withColumn("Century", (floor(col("IPO Year") / 100, 0) + 1).cast("int"))

window_spec = Window.partitionBy("Century").orderBy(
    col("Max Price").desc()
)

result = df_agg.withColumn(
    "Rank",
    rank().over(window_spec)
)

result.show(100)

+--------+---------+---------+---------+-----------------+-----------+----------+-------+----+
|IPO Year|Avg Price|Max Price|Min Price|   Max Market Cap|Stock Count|Min Volume|Century|Rank|
+--------+---------+---------+---------+-----------------+-----------+----------+-------+----+
|    1998|   229.34|   6325.4|     0.21| 1.33150596958E11|         38|       950|     20|   1|
|    1996|   200.65|  4526.23|     0.34|  3.8623064489E10|         30|      3846|     20|   2|
|    1986|   153.95|  1962.93|     1.68|2.954659903516E12|         27|      4282|     20|   3|
|    1995|   174.63|  1801.51|     2.59| 6.94333777674E11|         22|      6608|     20|   4|
|    1997|   198.85|  1773.51|   0.1277|2.662922440882E12|         40|      1877|     20|   5|
|    1999|   147.33|  1085.56|   0.1467|      5.016418E12|         39|      2491|     20|   6|
|    1983|    164.5|   858.26|     3.84|  1.9766275827E11|         11|       797|     20|   7|
|    1991|   143.55|   675.19|     8.47|       1.8

In [43]:
result.show(100)

+--------+---------+---------+---------+-----------------+-----------+----------+-------+----+
|IPO Year|Avg Price|Max Price|Min Price|   Max Market Cap|Stock Count|Min Volume|Century|Rank|
+--------+---------+---------+---------+-----------------+-----------+----------+-------+----+
|    1998|   229.34|   6325.4|     0.21| 1.33150596958E11|         38|       950|     20|   1|
|    1996|   200.65|  4526.23|     0.34|  3.8623064489E10|         30|      3846|     20|   2|
|    1986|   153.95|  1962.93|     1.68|2.954659903516E12|         27|      4282|     20|   3|
|    1995|   174.63|  1801.51|     2.59| 6.94333777674E11|         22|      6608|     20|   4|
|    1997|   198.85|  1773.51|   0.1277|2.662922440882E12|         40|      1877|     20|   5|
|    1999|   147.33|  1085.56|   0.1467|      5.016418E12|         39|      2491|     20|   6|
|    1983|    164.5|   858.26|     3.84|  1.9766275827E11|         11|       797|     20|   7|
|    1991|   143.55|   675.19|     8.47|       1.8

In [ ]:
# result.write \
#     .mode("overwrite") \
#     .parquet("output/stock_analysis")

Py4JJavaError: An error occurred while calling o970.parquet.
: java.util.concurrent.ExecutionException: Boxed Exception
	at scala.concurrent.impl.Promise$.scala$concurrent$impl$Promise$$resolve(Promise.scala:99)
	at scala.concurrent.impl.Promise$DefaultPromise.tryComplete(Promise.scala:288)
	at scala.concurrent.Promise.complete(Promise.scala:57)
	at scala.concurrent.Promise.complete$(Promise.scala:56)
	at scala.concurrent.impl.Promise$DefaultPromise.complete(Promise.scala:104)
	at scala.concurrent.Promise.failure(Promise.scala:109)
	at scala.concurrent.Promise.failure$(Promise.scala:109)
	at scala.concurrent.impl.Promise$DefaultPromise.failure(Promise.scala:104)
	at org.apache.spark.sql.execution.adaptive.ResultQueryStageExec.$anonfun$doMaterialize$2(QueryStageExec.scala:336)
	at java.base/java.util.concurrent.CompletableFuture.uniWhenComplete(CompletableFuture.java:863)
	at java.base/java.util.concurrent.CompletableFuture$UniWhenComplete.tryFire(CompletableFuture.java:841)

In [ ]:
# parquet_df = spark.read.parquet("output/stock_analysis")

# parquet_df.show()
# parquet_df.printSchema()